# 📈 Stock Data Analysis

這個 Notebook 專門用來調用 `database` 模組中的資料，並將其轉換為 Pandas DataFrame 以供分析。

### 前置準備
請確保您已安裝以下套件：
```bash
pip install pandas plotly jupyterlab
```

In [1]:
import pandas as pd
import sys
import os

# 1. 設定路徑：確保可以 import 專案根目錄的模組
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.append(current_dir)

try:
    from database.client import get_db_client
    print("✅ 成功載入 database 模組")
except ImportError as e:
    print(f"❌ 載入失敗: {e}")
    print("   請確認 database/client.py 存在且路徑正確。")

❌ 載入失敗: No module named 'postgres_client'
   請確認 database/client.py 存在且路徑正確。


In [3]:
from database.client import get_db_client

ModuleNotFoundError: No module named 'postgres_client'

In [ ]:
def get_table_as_df(table_name):
    """
    使用專案的 db client 讀取整張資料表
    """
    print(f"📥 正在讀取資料表: {table_name} ...")
    try:
        with get_db_client() as db:
            # 根據 crawlers/price.py 的用法，db.fetch 回傳 DataFrame
            df = db.fetch(table_name)
            print(f"✅ 讀取完成！共 {len(df)} 筆資料")
            return df
    except Exception as e:
        print(f"❌ 讀取錯誤: {e}")
        return pd.DataFrame()

## 1. 讀取台股資料 (TWStock)

In [ ]:
df_tw = get_table_as_df("tw_daily_prices")

# 預覽前 5 筆
df_tw.head()

## 2. 讀取美股資料 (USStock)

In [ ]:
df_us = get_table_as_df("us_daily_prices")

# 預覽前 5 筆
df_us.head()

## 3. 簡單視覺化範例 (Plotly)
這裡示範如何畫出 K 線圖。

In [ ]:
import plotly.graph_objects as go

def plot_stock(df, stock_id_or_ticker):
    if df.empty:
        print("⚠️ 資料集為空，無法繪圖")
        return
    
    # 判斷欄位名稱 (台股用 stock_id, 美股用 ticker)
    col_name = 'stock_id' if 'stock_id' in df.columns else 'ticker'
    
    # 篩選資料
    target_df = df[df[col_name] == stock_id_or_ticker].copy()
    
    if target_df.empty:
        print(f"⚠️ 找不到代號 {stock_id_or_ticker} 的資料")
        return
        
    # 確保日期格式
    target_df['date'] = pd.to_datetime(target_df['date'])
    target_df = target_df.sort_values('date')
    
    # 繪圖
    fig = go.Figure(data=[go.Candlestick(
        x=target_df['date'],
        open=target_df['open'],
        high=target_df['high'],
        low=target_df['low'],
        close=target_df['close'],
        name=stock_id_or_ticker
    )])
    
    fig.update_layout(
        title=f"{stock_id_or_ticker} 股價走勢",
        yaxis_title="Price",
        xaxis_title="Date",
        template="plotly_dark"
    )
    fig.show()

# 測試：畫出台積電 (2330) 或其他存在的股票
# 請確保資料庫內有該股票資料
plot_stock(df_tw, "2330")